In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
zip_path = "/content/drive/MyDrive/unet-data/unet-data.zip"  # zip adını güncelle
extract_path = "/content/ct-ich"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

# Yapıyı kontrol et
import os
for root, dirs, files in os.walk(extract_path):
    if files:
        print(f"{root}: {len(files)} dosya, örnek: {files[0]}")
    if root.count(os.sep) > 5:
        break

Mounted at /content/drive
/content/ct-ich/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1: 7 dosya, örnek: SHA256SUMS.txt
/content/ct-ich/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/ct_scans: 75 dosya, örnek: 114.nii
/content/ct-ich/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1/masks: 75 dosya, örnek: 114.nii


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/unet-data/"

# Klasördeki dosyaları listele
import os
print(os.listdir(zip_path))

['unet-data.zip']


In [ ]:
from pathlib import Path

BASE = "/content/ct-ich/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1"

files = list(Path(BASE + "/ct_scans").glob("*.nii"))
print(f"Toplam dosya: {len(files)}")
print(f"İlk 5: {[f.name for f in files[:5]]}")

Toplam dosya: 75
İlk 5: ['114.nii', '055.nii', '091.nii', '118.nii', '109.nii']


In [ ]:
%%capture
!pip install segmentation-models-pytorch nibabel
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import numpy as np
import nibabel as nib
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import torch

BASE = "/content/ct-ich/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1"

class CTICHDataset(Dataset):

    def __init__(self, base_dir, patient_ids, augment=False):
        self.ct_dir   = Path(base_dir) / "ct_scans"
        self.mask_dir = Path(base_dir) / "masks"
        self.augment  = augment

        # Slice bazlı liste: (hasta_id, slice_idx)
        self.slices = []
        for pid in patient_ids:
            ct_path   = self.ct_dir   / f"{pid}.nii"
            mask_path = self.mask_dir / f"{pid}.nii"
            if not ct_path.exists() or not mask_path.exists():
                continue
            vol = nib.load(str(ct_path)).get_fdata()
            n_slices = vol.shape[2]
            for i in range(n_slices):
                self.slices.append((pid, i))

        pos = 0
        for pid, i in self.slices:
            m = nib.load(str(self.mask_dir / f"{pid}.nii")).get_fdata()[:,:,i]
            if m.max() > 0:
                pos += 1
        print(f"Toplam: {len(self.slices)} slice | Kanamalı: {pos} | Negatif: {len(self.slices)-pos}")

    def __len__(self):
        return len(self.slices)

    def window(self, im, center, width):
        mn, mx = center - width//2, center + width//2
        return (np.clip(im, mn, mx) - mn) / (mx - mn)

    def __getitem__(self, idx):
        pid, slice_idx = self.slices[idx]

        ct   = nib.load(str(self.ct_dir   / f"{pid}.nii")).get_fdata()
        mask = nib.load(str(self.mask_dir / f"{pid}.nii")).get_fdata()

        sl = ct[:, :, slice_idx].astype(np.float32)
        mk = mask[:, :, slice_idx].astype(np.float32)

        # Pipeline ile AYNI 3-kanal windowing
        c1 = self.window(sl, 40,  80)    # Brain
        c2 = self.window(sl, 500, 3000)  # Bone
        c3 = self.window(sl, 175, 50)    # Subdural

        img = np.stack([c1, c2, c3], axis=0).astype(np.float32)
        mk  = (mk > 0.5).astype(np.float32)[np.newaxis]

        # Augmentation
        if self.augment and np.random.rand() > 0.5:
            img = np.flip(img, axis=2).copy()
            mk  = np.flip(mk,  axis=2).copy()
        if self.augment and np.random.rand() > 0.5:
            img = np.flip(img, axis=1).copy()
            mk  = np.flip(mk,  axis=1).copy()

        return torch.tensor(img), torch.tensor(mk)

# Train/val split (hasta bazlı — slice bazlı değil, önemli!)
all_ids = sorted([f.stem for f in Path(BASE + "/ct_scans").glob("*.nii")])
print(f"İlk 5 ID: {all_ids[:5]}")
np.random.seed(42)
np.random.shuffle(all_ids)  # string shuffle sorun çıkarmaz
n_val = int(len(all_ids) * 0.2)
val_ids   = all_ids[:n_val]
train_ids = all_ids[n_val:]

print(f"Train hastaları: {len(train_ids)} | Val hastaları: {len(val_ids)}")
train_ds = CTICHDataset(BASE, train_ids, augment=True)
val_ds   = CTICHDataset(BASE, val_ids,   augment=False)

İlk 5 ID: ['049', '050', '051', '052', '053']
Train hastaları: 60 | Val hastaları: 15
Toplam: 2246 slice | Kanamalı: 247 | Negatif: 1999
Toplam: 568 slice | Kanamalı: 71 | Negatif: 497


In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(
    encoder_name="resnet34",
    in_channels=3,
    classes=1,
    encoder_weights="imagenet"
).to(device)

class DiceBCELoss(nn.Module):
    def forward(self, pred, target):
        pred_s = torch.sigmoid(pred)
        inter  = (pred_s * target).sum(dim=(2,3))
        dice   = 1 - (2*inter + 1) / (pred_s.sum(dim=(2,3)) + target.sum(dim=(2,3)) + 1)
        bce = nn.functional.binary_cross_entropy_with_logits(
            pred, target, pos_weight=torch.tensor([8.0]).to(pred.device)
        )
        return dice.mean() + bce

criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

print("✅ Model hazır:", sum(p.numel() for p in model.parameters())/1e6, "M parametre")

✅ Model hazır: 24.436369 M parametre


In [ ]:
import nibabel as nib
from torch.utils.data import DataLoader, WeightedRandomSampler

# Sampler için ağırlıklar
weights = []
for pid, i in train_ds.slices:
    m = nib.load(str(Path(BASE) / "masks" / f"{pid}.nii")).get_fdata()[:,:,i]
    weights.append(8.0 if m.max() > 0 else 1.0)

sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,   num_workers=2, pin_memory=True)

def dice_score(pred, target, thr=0.5):
    p = (torch.sigmoid(pred) > thr).float()
    inter = (p * target).sum()
    return (2*inter + 1) / (p.sum() + target.sum() + 1)

EPOCHS    = 30
best_dice = 0.0
save_path = "/content/drive/MyDrive/unet_ct_ich_best.pth"

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    t_loss = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        t_loss += loss.item()

    # ── Validate ──
    model.eval()
    v_loss, v_dice = 0, 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            v_loss += criterion(preds, masks).item()
            v_dice += dice_score(preds, masks).item()

    v_dice /= len(val_loader)
    v_loss /= len(val_loader)
    t_loss /= len(train_loader)
    scheduler.step(v_loss)

    saved = v_dice > best_dice
    if saved:
        best_dice = v_dice
        torch.save(model.state_dict(), save_path)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Val Dice: {v_dice:.4f}"
          + (" ✅ SAVED" if saved else ""))

print(f"\n🏆 Eğitim tamamlandı. En iyi Val Dice: {best_dice:.4f}")
print(f"💾 Model kaydedildi: {save_path}")

Epoch 01/30 | Train: 1.1812 | Val Loss: 1.0795 | Val Dice: 0.2897 ✅ SAVED
Epoch 02/30 | Train: 0.9642 | Val Loss: 1.0213 | Val Dice: 0.2702
Epoch 03/30 | Train: 0.8405 | Val Loss: 1.0132 | Val Dice: 0.1274
Epoch 04/30 | Train: 0.7550 | Val Loss: 0.9623 | Val Dice: 0.4492 ✅ SAVED
Epoch 05/30 | Train: 0.7235 | Val Loss: 0.9620 | Val Dice: 0.3923
Epoch 06/30 | Train: 0.6749 | Val Loss: 0.9605 | Val Dice: 0.6380 ✅ SAVED
Epoch 07/30 | Train: 0.6634 | Val Loss: 0.9459 | Val Dice: 0.4720
Epoch 08/30 | Train: 0.6530 | Val Loss: 0.9676 | Val Dice: 0.6557 ✅ SAVED
Epoch 09/30 | Train: 0.6025 | Val Loss: 0.4697 | Val Dice: 0.4990
Epoch 10/30 | Train: 0.4175 | Val Loss: 0.5069 | Val Dice: 0.5333
Epoch 11/30 | Train: 0.3455 | Val Loss: 0.1490 | Val Dice: 0.7508 ✅ SAVED
Epoch 12/30 | Train: 0.4184 | Val Loss: 0.1794 | Val Dice: 0.6528
Epoch 13/30 | Train: 0.3527 | Val Loss: 0.1389 | Val Dice: 0.6866
Epoch 14/30 | Train: 0.2440 | Val Loss: 0.1113 | Val Dice: 0.7647 ✅ SAVED
Epoch 15/30 | Train: 0.2991 

### Model Performansını Değerlendirme

Kaydedilen en iyi modeli yükleyip doğrulama seti üzerinde 'Dice Skoru'nu hesaplayalım.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install segmentation-models-pytorch nibabel

import torch
import segmentation_models_pytorch as smp
import os
import numpy as np
import nibabel as nib
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import zipfile

# --- File Extraction (Moved from uHOONhmSybA7 for robustness) ---
zip_path = "/content/drive/MyDrive/unet-data/unet-data.zip"
extract_path = "/content/ct-ich"

# Ensure the extraction directory exists
os.makedirs(extract_path, exist_ok=True)

# Only extract if the target directory is empty or doesn't exist
if not os.path.exists(os.path.join(extract_path, "computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1")):
    print(f"Extracting data from {zip_path} to {extract_path}...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print("Extraction complete.")
else:
    print(f"Data already extracted to {extract_path}.")
# -----------------------------------------------------------------

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define model architecture (from bnMsfEMO1zVd)
model = smp.Unet(
    encoder_name="resnet34",
    in_channels=3,
    classes=1,
    encoder_weights="imagenet"
).to(device)

# Define save_path (from iyZW1aFl12MV)
save_path = "/content/drive/MyDrive/unet_ct_ich_best.pth"

# Check if the model file exists
if not os.path.exists(save_path):
    print(f"Hata: Model dosyası bulunamadı: {save_path}")
    print("Lütfen aşağıdaki çıktıdan doğru yolu ve dosya adını kontrol edin:")
    print("\n/content/drive/MyDrive/ klasörünün içeriği:")
    !ls -F "/content/drive/MyDrive/"
    raise FileNotFoundError(f"Model dosyası bulunamadı: {save_path}")

# Define dice_score function (from iyZW1aFl12MV)
def dice_score(pred, target, thr=0.5):
    p = (torch.sigmoid(pred) > thr).float()
    inter = (p * target).sum()
    return (2*inter + 1) / (p.sum() + target.sum() + 1)

# --- Dataset and DataLoader definitions (from -xGm_bvd1wKN and iyZW1aFl12MV) ---
BASE = "/content/ct-ich/computed-tomography-images-for-intracranial-hemorrhage-detection-and-segmentation-1.3.1"

class CTICHDataset(Dataset):
    def __init__(self, base_dir, patient_ids, augment=False):
        self.ct_dir   = Path(base_dir) / "ct_scans"
        self.mask_dir = Path(base_dir) / "masks"
        self.augment  = augment

        self.slices = []
        for pid in patient_ids:
            ct_path   = self.ct_dir   / f"{pid}.nii"
            mask_path = self.mask_dir / f"{pid}.nii"
            if not ct_path.exists() or not mask_path.exists():
                print(f"Uyarı: {pid}.nii için CT veya maske dosyası bulunamadı. Atlanıyor.")
                continue
            try:
                vol = nib.load(str(ct_path)).get_fdata()
                n_slices = vol.shape[2]
                for i in range(n_slices):
                    self.slices.append((pid, i))
            except Exception as e:
                print(f"Uyarı: {pid}.nii dosyası yüklenirken hata oluştu: {e}. Atlanıyor.")

    def __len__(self):
        return len(self.slices)

    def window(self, im, center, width):
        mn, mx = center - width//2, center + width//2
        return (np.clip(im, mn, mx) - mn) / (mx - mn)

    def __getitem__(self, idx):
        pid, slice_idx = self.slices[idx]

        ct   = nib.load(str(self.ct_dir   / f"{pid}.nii")).get_fdata()
        mask = nib.load(str(self.mask_dir / f"{pid}.nii")).get_fdata()

        sl = ct[:, :, slice_idx].astype(np.float32)
        mk = mask[:, :, slice_idx].astype(np.float32)

        c1 = self.window(sl, 40,  80)    # Brain
        c2 = self.window(sl, 500, 3000)  # Bone
        c3 = self.window(sl, 175, 50)    # Subdural

        img = np.stack([c1, c2, c3], axis=0).astype(np.float32)
        mk  = (mk > 0.5).astype(np.float32)[np.newaxis]

        if self.augment and np.random.rand() > 0.5:
            img = np.flip(img, axis=2).copy()
            mk  = np.flip(mk,  axis=2).copy()
        if self.augment and np.random.rand() > 0.5:
            img = np.flip(img, axis=1).copy()
            mk  = np.flip(mk,  axis=1).copy()

        return torch.tensor(img), torch.tensor(mk)

# Train/val split (hasta bazlı — slice bazlı değil, önemli!)
all_ids = sorted([f.stem for f in Path(BASE + "/ct_scans").glob("*.nii")])

# Add a check for empty all_ids
if not all_ids:
    print(f"Hata: {BASE + '/ct_scans'} dizininde .nii dosyası bulunamadı. Lütfen veri yolunu kontrol edin.")
    # List contents of BASE directory for debugging
    print(f"\n{BASE} klasörünün içeriği:")
    !ls -F "{BASE}"
    raise RuntimeError("Dataset dosyaları bulunamadı, val_loader oluşturulamıyor.")

np.random.seed(42)
np.random.shuffle(all_ids)
n_val = int(len(all_ids) * 0.2)
val_ids   = all_ids[:n_val]
train_ids = all_ids[n_val:]

train_ds = CTICHDataset(BASE, train_ids, augment=True)
val_ds   = CTICHDataset(BASE, val_ids,   augment=False)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True) # Sampler'ı eğitim için kullanmıyoruz, sadece değerlendirme
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,   num_workers=2, pin_memory=True)

# En iyi modeli yükle
model.load_state_dict(torch.load(save_path))
model.eval() # Modeli değerlendirme moduna al

total_dice_score = 0.0
with torch.no_grad():
    for imgs, masks in val_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        total_dice_score += dice_score(preds, masks).item()

average_dice_score = total_dice_score / len(val_loader)
print(f"Yüklenen modelin doğrulama setindeki ortalama Dice Skoru: {average_dice_score:.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting data from /content/drive/MyDrive/unet-data/unet-data.zip to /content/ct-ich...
Extraction complete.
Yüklenen modelin doğrulama setindeki ortalama Dice Skoru: 0.8192


### Detaylı Performans Metrikleri Hesaplama

Modelin performansını daha detaylı değerlendirmek için True Positives (TP), False Positives (FP), True Negatives (TN), False Negatives (FN), Sensitivity (Recall), Specificity, Precision ve F1-Score gibi metrikleri hesaplayalım.

In [ ]:
def calculate_metrics(pred, target, thr=0.5):
    pred = (torch.sigmoid(pred) > thr).float()

    tp = (pred * target).sum()  # True Positives
    fp = (pred * (1 - target)).sum() # False Positives
    fn = ((1 - pred) * target).sum() # False Negatives
    tn = ((1 - pred) * (1 - target)).sum() # True Negatives

    return tp, fp, fn, tn

# Reset metrics
total_tp = 0
total_fp = 0
total_fn = 0
total_tn = 0

model.eval()
with torch.no_grad():
    for imgs, masks in val_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        preds = model(imgs)
        tp, fp, fn, tn = calculate_metrics(preds, masks)

        total_tp += tp
        total_fp += fp
        total_fn += fn
        total_tn += tn

# Calculate overall metrics
sensitivity = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
specificity = total_tn / (total_tn + total_fp) if (total_tn + total_fp) > 0 else 0
precision   = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
f1_score    = (2 * precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

print(f"\n--- Detaylı Performans Metrikleri ---")
print(f"True Positives (TP): {total_tp.item():.0f}")
print(f"False Positives (FP): {total_fp.item():.0f}")
print(f"False Negatives (FN): {total_fn.item():.0f}")
print(f"True Negatives (TN): {total_tn.item():.0f}")
print(f"Sensitivity (Recall): {sensitivity.item():.4f}")
print(f"Specificity: {specificity.item():.4f}")
print(f"Precision: {precision.item():.4f}")
print(f"F1-Score: {f1_score.item():.4f}")


--- Detaylı Performans Metrikleri ---
True Positives (TP): 33198
False Positives (FP): 13826
False Negatives (FN): 31803
True Negatives (TN): 148818960
Sensitivity (Recall): 0.5107
Specificity: 0.9999
Precision: 0.7060
F1-Score: 0.5927
